# Road PCI Intelligence — Step 3C: Cross-Domain RDD2022ES Validation Check

This notebook evaluates **Model A (Masked Loss Baseline)** and **Model B (Naive Loss Baseline)** on the **RDD2022ES validation split** (2,560 images) without retraining:
- **Disclaimer**: RDD bounding boxes are loose, weak labels covering entire road sections containing distress. They are **not pixel-level segmentation masks**, so these metrics are **not Dice scores** and are **not directly comparable to earlier segmentation numbers**.
- **Inference Configuration**: Full image padded to multiple of 32, sigmoid activation, decision threshold 0.5.
- **Box Classes**: Classes 0–5 = Cracks, Classes 6–7 = Potholes.


In [ ]:
# ==============================================================================
# 1. SETUP ENVIRONMENT, DEPENDENCIES & REPO REPLICATION
# ==============================================================================
import os
import sys
import subprocess
from pathlib import Path

print("Installing segmentation-models-pytorch and albumentations...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch==0.5.0", "albumentations==2.0.8"], check=True)

repo_dir = Path('/kaggle/working/repo')
if not repo_dir.exists():
    print("Cloning GitHub repository...")
    subprocess.run(["git", "clone", "https://github.com/skilloutofbounds/road-pci-intelligence.git", str(repo_dir)], check=True)
else:
    print("Pulling latest GitHub changes...")
    subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

import torch
import albumentations as A
import segmentation_models_pytorch as smp

from src.train import seed_everything, log_library_versions

seed_everything(42)
log_library_versions()

assert torch.cuda.is_available(), "CRITICAL: CUDA is required for efficient RDD inference!"
device = torch.device('cuda')
print(f"CUDA Device: {torch.cuda.get_device_name(0)}")


In [ ]:
# ==============================================================================
# 2. LOCATE CHECKPOINTS & RESOLVE RDD DATASET ROOT & MANIFEST
# ==============================================================================
from pathlib import Path
import pandas as pd

k_input = Path('/kaggle/input')

# Locate Model A best.pt
cand_a = list(k_input.rglob('*run_a*/**/best.pt')) + list(k_input.rglob('*masked*/**/best.pt'))
assert len(cand_a) > 0, f"Could not find Model A best.pt in /kaggle/input! Candidates: {cand_a}"
model_a_pt = cand_a[0]
print(f"Model A checkpoint: {model_a_pt}")

# Locate Model B best.pt
cand_b = list(k_input.rglob('*run_b*/**/best.pt')) + list(k_input.rglob('*naive*/**/best.pt'))
assert len(cand_b) > 0, f"Could not find Model B best.pt in /kaggle/input! Candidates: {cand_b}"
model_b_pt = cand_b[0]
print(f"Model B checkpoint: {model_b_pt}")

# Resolve RDD dataset root
candidates = [
    Path('/kaggle/input/datasets/juusos/rdd2022es'),
    Path('/kaggle/input/rdd2022es'),
    Path('data/rdd2022es')
]
dataset_root = None
for c in candidates:
    if c.exists() and c.is_dir():
        dataset_root = c.resolve()
        break

if dataset_root is None:
    for d in k_input.rglob('rdd2022es'):
        if d.is_dir():
            dataset_root = d.resolve()
            break

assert dataset_root is not None, "Could not find RDD dataset root in /kaggle/input!"
print(f"RDD dataset root: {dataset_root}")

# Load RDD manifest
manifest_path = repo_dir / 'manifests/rdd2022es_manifest.csv'
assert manifest_path.exists(), f"Missing RDD manifest: {manifest_path}"
df_rdd = pd.read_csv(manifest_path)
print(f"Loaded RDD manifest: {len(df_rdd):,} total records")

# Filter strictly to VAL split
df_val = df_rdd[df_rdd['split'] == 'val'].copy().reset_index(drop=True)
print(f"Filtered strictly to VAL split: {len(df_val):,} images (Target: 2,560)")
assert len(df_val) == 2560, f"Expected 2,560 val images, found {len(df_val)}"


In [ ]:
# ==============================================================================
# 3. BUILD MODELS & LOAD WEIGHTS
# ==============================================================================
def load_eval_model(weights_path: Path) -> smp.Unet:
    model = smp.Unet(
        encoder_name='resnet34',
        encoder_weights=None,
        in_channels=3,
        classes=2
    )
    state = torch.load(weights_path, map_location=device)
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    return model

print("Loading Model A (Masked)...")
model_a = load_eval_model(model_a_pt)
print("Loading Model B (Naive)...")
model_b = load_eval_model(model_b_pt)
print("[OK] Both models loaded successfully.")


In [ ]:
# ==============================================================================
# 4. RDD LABEL PARSER & EVALUATION PIPELINE
# ==============================================================================
import cv2
import numpy as np
from PIL import Image
from torch.cuda.amp import autocast
from collections import defaultdict
import time

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def parse_rdd_boxes(txt_path: Path, width: int, height: int):
    """
    Parses YOLO-format bounding boxes from RDD annotation file.
    Returns list of dicts: [{'class_id': int, 'box': (ymin, ymax, xmin, xmax), 'area': int}]
    """
    boxes = []
    if not txt_path.exists():
        return boxes
        
    with open(txt_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cid = int(float(parts[0]))
                xc = float(parts[1]) * width
                yc = float(parts[2]) * height
                w = float(parts[3]) * width
                h = float(parts[4]) * height
                
                xmin = max(0, int(round(xc - w / 2.0)))
                ymin = max(0, int(round(yc - h / 2.0)))
                xmax = min(width, int(round(xc + w / 2.0)))
                ymax = min(height, int(round(yc + h / 2.0)))
                
                if xmax > xmin and ymax > ymin:
                    boxes.append({
                        'class_id': cid,
                        'box': (ymin, ymax, xmin, xmax),
                        'area': (ymax - ymin) * (xmax - xmin)
                    })
    return boxes

@torch.no_grad()
def infer_single_image(model, img_rgb: np.ndarray) -> np.ndarray:
    """
    Inference with padding to multiple of 32.
    Returns probability map of shape (2, H, W).
    """
    h, w = img_rgb.shape[:2]
    pad_h = int(np.ceil(h / 32.0) * 32) - h
    pad_w = int(np.ceil(w / 32.0) * 32) - w
    
    img_padded = cv2.copyMakeBorder(img_rgb, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)
    norm = ((img_padded / 255.0) - IMAGENET_MEAN) / IMAGENET_STD
    tensor = torch.from_numpy(norm).permute(2, 0, 1).unsqueeze(0).float().to(device)
    
    with autocast():
        logits = model(tensor)
        probs = torch.sigmoid(logits)[0, :, :h, :w].cpu().numpy()
        
    return probs

print("Starting inference across 2,560 RDD validation images...")
t0 = time.time()

records = []
for idx, row in df_val.iterrows():
    img_p = dataset_root / row['image_path']
    txt_p = dataset_root / row['mask_path_or_label_path'] if row['mask_path_or_label_path'] else Path('')
    
    with Image.open(img_p) as pil_img:
        img_rgb = np.array(pil_img.convert('RGB'))
    h, w = img_rgb.shape[:2]
    n_pixels = h * w
    
    boxes = parse_rdd_boxes(txt_p, w, h)
    
    # Classify boxes
    crack_boxes = [b for b in boxes if 0 <= b['class_id'] <= 5]
    pothole_boxes = [b for b in boxes if 6 <= b['class_id'] <= 7]
    
    # Run inference for Model A and Model B
    probs_a = infer_single_image(model_a, img_rgb)
    probs_b = infer_single_image(model_b, img_rgb)
    
    # Binary predictions at threshold 0.5
    pred_a_crack = (probs_a[0] >= 0.5)
    pred_a_pothole = (probs_a[1] >= 0.5)
    
    pred_b_crack = (probs_b[0] >= 0.5)
    pred_b_pothole = (probs_b[1] >= 0.5)
    
    records.append({
        'index': idx,
        'image_path': str(img_p),
        'boxes': boxes,
        'crack_boxes': crack_boxes,
        'pothole_boxes': pothole_boxes,
        'n_pixels': n_pixels,
        # Predictions Model A
        'a_crack_fg_frac': float(pred_a_crack.sum() / n_pixels),
        'a_pothole_fg_frac': float(pred_a_pothole.sum() / n_pixels),
        'a_pred_crack': pred_a_crack,
        'a_pred_pothole': pred_a_pothole,
        # Predictions Model B
        'b_crack_fg_frac': float(pred_b_crack.sum() / n_pixels),
        'b_pothole_fg_frac': float(pred_b_pothole.sum() / n_pixels),
        'b_pred_crack': pred_b_crack,
        'b_pred_pothole': pred_b_pothole,
    })
    
    if (idx + 1) % 500 == 0 or (idx + 1) == len(df_val):
        print(f"Processed {idx + 1:4d} / {len(df_val)} images ({time.time() - t0:.1f}s)...")

t_elapsed = time.time() - t0
print(f"Completed RDD validation inference in {t_elapsed:.2f}s ({len(df_val)/t_elapsed:.1f} img/s)")


In [ ]:
# ==============================================================================
# 5. COMPUTE & REPORT DETAILED RDD CROSS-DOMAIN METRICS
# ==============================================================================
import numpy as np

# A. IMAGES WITH ZERO BOXES
zero_box_records = [r for r in records if len(r['boxes']) == 0]
n_zero = len(zero_box_records)
print("=" * 90)
print(f"A. IMAGES WITH ZERO BOXES (N = {n_zero:,} images)")
print("=" * 90)
print(f"{'Model':<8} | {'Channel':<9} | {'Flag > 0.1%':<14} | {'Flag > 0.5%':<14} | {'Flag > 1.0%':<14} | {'Mean FG Frac':<14}")
print("-" * 90)

for m_key, m_name in [('a', 'Model A (Masked)'), ('b', 'Model B (Naive)')]:
    for ch_name in ['crack', 'pothole']:
        fracs = np.array([r[f'{m_key}_{ch_name}_fg_frac'] for r in zero_box_records])
        r01 = np.mean(fracs > 0.001) * 100
        r05 = np.mean(fracs > 0.005) * 100
        r10 = np.mean(fracs > 0.010) * 100
        m_frac = np.mean(fracs) * 100
        print(f"{m_key.upper():<8} | {ch_name:<9} | {r01:6.2f}% ({np.sum(fracs > 0.001):4d}) | {r05:6.2f}% ({np.sum(fracs > 0.005):4d}) | {r10:6.2f}% ({np.sum(fracs > 0.010):4d}) | {m_frac:6.4f}%")
print("-" * 90)

# B. IMAGES WITH DISTRESS BOXES (HIT RATE & BOX COVERAGE)
print("\n" + "=" * 90)
print("B. HIT RATE & BOX PIXEL COVERAGE ON IMAGES WITH GROUND-TRUTH BOXES")
print("=" * 90)

def compute_box_metrics(target_records, box_key, pred_key, cid_filter=None):
    img_hits = []
    box_fracs = []
    
    for r in target_records:
        boxes = [b for b in r[box_key] if cid_filter is None or b['class_id'] == cid_filter]
        if not boxes:
            continue
        
        pred_mask = r[pred_key]
        img_has_hit = False
        
        for b in boxes:
            ymin, ymax, xmin, xmax = b['box']
            area = b['area']
            p_inside = pred_mask[ymin:ymax, xmin:xmax].sum()
            frac = float(p_inside / area)
            box_fracs.append(frac)
            if frac >= 0.05:
                img_has_hit = True
                
        img_hits.append(img_has_hit)
        
    n_imgs = len(img_hits)
    hit_rate = np.mean(img_hits) * 100 if n_imgs > 0 else 0.0
    mean_box_frac = np.mean(box_fracs) * 100 if box_fracs else 0.0
    return n_imgs, len(box_fracs), hit_rate, mean_box_frac

# Potholes (classes 6-7)
pothole_records = [r for r in records if len(r['pothole_boxes']) > 0]
print(f"\n--- Pothole Channel vs Pothole Boxes (N = {len(pothole_records)} images) ---")
print(f"{'Model':<8} | {'Category':<18} | {'Images':<8} | {'Boxes':<8} | {'Hit Rate (>=5%)':<18} | {'Mean Box Frac':<14}")
print("-" * 90)
for m_key in ['a', 'b']:
    n_i, n_b, hr, mbf = compute_box_metrics(pothole_records, 'pothole_boxes', f'{m_key}_pred_pothole')
    print(f"{m_key.upper():<8} | {'All Potholes (6-7)':<18} | {n_i:<8} | {n_b:<8} | {hr:6.2f}% ({int(round(hr*n_i/100)):3d}/{n_i}) | {mbf:6.2f}%")
    for cid in [6, 7]:
        n_ic, n_bc, hrc, mbfc = compute_box_metrics(pothole_records, 'pothole_boxes', f'{m_key}_pred_pothole', cid_filter=cid)
        if n_ic > 0:
            print(f"{m_key.upper():<8} | {f'  Class {cid}':<18} | {n_ic:<8} | {n_bc:<8} | {hrc:6.2f}% ({int(round(hrc*n_ic/100)):3d}/{n_ic}) | {mbfc:6.2f}%")

# Cracks (classes 0-5)
crack_records = [r for r in records if len(r['crack_boxes']) > 0]
print(f"\n--- Crack Channel vs Crack Boxes (N = {len(crack_records)} images) ---")
print(f"{'Model':<8} | {'Category':<18} | {'Images':<8} | {'Boxes':<8} | {'Hit Rate (>=5%)':<18} | {'Mean Box Frac':<14}")
print("-" * 90)
for m_key in ['a', 'b']:
    n_i, n_b, hr, mbf = compute_box_metrics(crack_records, 'crack_boxes', f'{m_key}_pred_crack')
    print(f"{m_key.upper():<8} | {'All Cracks (0-5)':<18} | {n_i:<8} | {n_b:<8} | {hr:6.2f}% ({int(round(hr*n_i/100)):3d}/{n_i}) | {mbf:6.2f}%")
    for cid in range(6):
        n_ic, n_bc, hrc, mbfc = compute_box_metrics(crack_records, 'crack_boxes', f'{m_key}_pred_crack', cid_filter=cid)
        if n_ic > 0:
            print(f"{m_key.upper():<8} | {f'  Class {cid}':<18} | {n_ic:<8} | {n_bc:<8} | {hrc:6.2f}% ({int(round(hrc*n_ic/100)):3d}/{n_ic}) | {mbfc:6.2f}%")

# C. CROSS-CHANNEL CHECK
only_crack_records = [r for r in records if len(r['crack_boxes']) > 0 and len(r['pothole_boxes']) == 0]
only_pothole_records = [r for r in records if len(r['pothole_boxes']) > 0 and len(r['crack_boxes']) == 0]

print("\n" + "=" * 90)
print(f"C. CROSS-CHANNEL AUDIT")
print(f"  • Images with ONLY Crack boxes:   {len(only_crack_records):,} images")
print(f"  • Images with ONLY Pothole boxes: {len(only_pothole_records):,} images")
print("=" * 90)
print(f"{'Model':<8} | {'Condition':<25} | {'Evaluated Ch':<14} | {'Flag > 0.1%':<14} | {'Flag > 0.5%':<14} | {'Mean FG Frac':<14}")
print("-" * 90)

for m_key in ['a', 'b']:
    # Pothole-channel on only-crack images
    pot_fracs = np.array([r[f'{m_key}_pothole_fg_frac'] for r in only_crack_records])
    r01_p = np.mean(pot_fracs > 0.001) * 100
    r05_p = np.mean(pot_fracs > 0.005) * 100
    m_p = np.mean(pot_fracs) * 100
    print(f"{m_key.upper():<8} | {'Only Crack Boxes':<25} | {'Pothole':<14} | {r01_p:6.2f}% ({np.sum(pot_fracs > 0.001):4d}) | {r05_p:6.2f}% ({np.sum(pot_fracs > 0.005):4d}) | {m_p:6.4f}%")
    
    # Crack-channel on only-pothole images
    crk_fracs = np.array([r[f'{m_key}_crack_fg_frac'] for r in only_pothole_records])
    r01_c = np.mean(crk_fracs > 0.001) * 100
    r05_c = np.mean(crk_fracs > 0.005) * 100
    m_c = np.mean(crk_fracs) * 100
    print(f"{m_key.upper():<8} | {'Only Pothole Boxes':<25} | {'Crack':<14} | {r01_c:6.2f}% ({np.sum(crk_fracs > 0.001):4d}) | {r05_c:6.2f}% ({np.sum(crk_fracs > 0.005):4d}) | {m_c:6.4f}%")
print("-" * 90)


In [ ]:
# ==============================================================================
# 6. VISUALIZATION: 12 REPRESENTATIVE VALIDATION IMAGES (SEED 42)
# ==============================================================================
import matplotlib.pyplot as plt
import random

rng = random.Random(42)

# Select 4 clean images (0 boxes)
clean_candidates = [r for r in records if len(r['boxes']) == 0]
selected_clean = rng.sample(clean_candidates, min(4, len(clean_candidates)))

# Select 4 pothole images (classes 6 or 7)
pothole_candidates = [r for r in records if len(r['pothole_boxes']) > 0]
selected_potholes = rng.sample(pothole_candidates, min(4, len(pothole_candidates)))

# Select 4 crack images with alligator (class 2) or longitudinal (class 0)
crack_candidates = [r for r in records if any(b['class_id'] in [0, 2] for b in r['crack_boxes'])]
selected_cracks = rng.sample(crack_candidates, min(4, len(crack_candidates)))

selected_12 = (
    [('Clean', r) for r in selected_clean] +
    [('Pothole', r) for r in selected_potholes] +
    [('Crack (D00/D20)', r) for r in selected_cracks]
)

fig, axes = plt.subplots(12, 3, figsize=(15, 36))

for row_idx, (cat_name, r) in enumerate(selected_12):
    # Load original image
    with Image.open(r['image_path']) as pil_img:
        img_rgb = np.array(pil_img.convert('RGB'))
    h, w = img_rgb.shape[:2]
    
    # Draw GT boxes on image copy
    img_gt = img_rgb.copy()
    for b in r['boxes']:
        ymin, ymax, xmin, xmax = b['box']
        cid = b['class_id']
        color = (0, 100, 255) if cid in [6, 7] else (255, 50, 50)
        cv2.rectangle(img_gt, (xmin, ymin), (xmax, ymax), color, 2)
        cv2.putText(img_gt, f"c{cid}", (xmin, max(15, ymin - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        
    stem = Path(r['image_path']).name
    if len(stem) > 22:
        stem = stem[:19] + "..."
        
    # Col 0: Image + GT Boxes
    axes[row_idx, 0].imshow(img_gt)
    axes[row_idx, 0].set_title(f"[{cat_name}] {stem} (GT Boxes: {len(r['boxes'])})", fontsize=9, fontweight='bold')
    axes[row_idx, 0].axis('off')
    
    # Col 1: Model A prediction overlay (Crack: Red, Pothole: Blue)
    overlay_a = img_rgb.copy()
    mask_a_crack = r['a_pred_crack']
    mask_a_pot = r['a_pred_pothole']
    overlay_a[mask_a_crack] = [230, 20, 20]
    overlay_a[mask_a_pot] = [20, 100, 240]
    axes[row_idx, 1].imshow(overlay_a)
    axes[row_idx, 1].set_title(f"Model A (Masked) | Crk FG: {r['a_crack_fg_frac']*100:.2f}% | Pot FG: {r['a_pothole_fg_frac']*100:.2f}%", fontsize=9)
    axes[row_idx, 1].axis('off')
    
    # Col 2: Model B prediction overlay (Crack: Red, Pothole: Blue)
    overlay_b = img_rgb.copy()
    mask_b_crack = r['b_pred_crack']
    mask_b_pot = r['b_pred_pothole']
    overlay_b[mask_b_crack] = [230, 20, 20]
    overlay_b[mask_b_pot] = [20, 100, 240]
    axes[row_idx, 2].imshow(overlay_b)
    axes[row_idx, 2].set_title(f"Model B (Naive) | Crk FG: {r['b_crack_fg_frac']*100:.2f}% | Pot FG: {r['b_pothole_fg_frac']*100:.2f}%", fontsize=9)
    axes[row_idx, 2].axis('off')

plt.subplots_adjust(top=0.98, bottom=0.01, hspace=0.25, wspace=0.08)
out_fig = Path('/kaggle/working/rdd_cross_domain_12_val.png')
plt.savefig(out_fig, dpi=120, bbox_inches='tight')
plt.close()
print(f"\n[OK] Saved 12-image visualization overlay figure to {out_fig}")
